Notebook: feature_engineering.ipynb  
Project: EcoPackAI  

This notebook creates engineered sustainability and performance features:
- CO₂ Impact Index (CII)
- Cost Efficiency Index (CEI)
- Material Suitability Score (MSS)


In [1]:
# cell 1 Environment setup
import pandas as pd
import numpy as np


In [2]:
#cell 2 Data loading
material = pd.read_csv(r'C:\Users\Ranjit\OneDrive\Desktop\AI-Powered-Sustainable-Packaging-Recommendation-System\ml\data\final\processed\material_cleaned.csv')
print("material and product data loaded successfully")
print(material.shape)
material.head()


material and product data loaded successfully
(403, 23)


,material_id,packaging_type,material_type,suitable_product_categories,recommended_packaging_use_cases,supplier_region,recyclability_,recyclability_category,recycled_content_,reusability_,...,co2_emission_per_kg_estimated_,waste_reduction_impact_,sustainability_target_progress_,strength,moisture_resistance_score,thermal_resistance_score,cost_per_unit_usd_,annual_usage_units_,total_material_weight_tons_,supplier_sustainability_compliance_
0,MAT_0001,Cardboard Boxes,Cardboard,"E-commerce, Food & Beverage, Consumer Goods, A...",Last-mile delivery and primary e-commerce pack...,EMEA,98,High,79.0,49.0,...,0.54,61.0,88.0,6.0,5.0,4.0,2.24,101260.0,790.0,85.0
1,MAT_0002,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,APAC,100,High,93.0,31.0,...,0.32,68.0,98.0,3.0,3.0,3.0,1.82,73961.0,545.0,88.0
2,MAT_0003,Steel Racks & Containers,Steel,"Heavy Industrial Components, High-Security Goods","Secure, high-load international shipping and l...",AMERICAS,87,High,78.0,100.0,...,3.25,85.0,79.0,8.0,9.0,9.0,25.00,14649.0,4979.0,88.0
3,MAT_0004,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,ROW,100,High,89.0,31.0,...,0.31,69.0,87.0,1.0,1.0,4.0,2.43,75036.0,656.0,81.0
4,MAT_0005,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,LATAM,100,High,92.0,28.0,...,0.31,61.0,89.0,3.0,2.0,3.0,1.69,69209.0,549.0,81.0


In [3]:
#cell 3 checking missing values
material.isnull().sum()

material_id                            0
packaging_type                         0
material_type                          0
suitable_product_categories            0
recommended_packaging_use_cases        0
supplier_region                        0
recyclability_                         0
recyclability_category                 0
recycled_content_                      0
reusability_                           0
biodegradation_time_days_              0
end_of_life_disposal_                  0
carbon_footprint_kg_co2_unit_          0
co2_emission_per_kg_estimated_         0
waste_reduction_impact_                0
sustainability_target_progress_        0
strength                               0
moisture_resistance_score              0
thermal_resistance_score               0
cost_per_unit_usd_                     0
annual_usage_units_                    0
total_material_weight_tons_            0
supplier_sustainability_compliance_    0
dtype: int64

In [4]:
# cell 4- Safe Min-Max Normalization
def min_max_safe(series):
    if series.max() == series.min():
        return np.zeros(len(series))
    return (series - series.min()) / (series.max() - series.min())


In [6]:
# cell 5 — CO₂ Impact Index 
co2_norm = 1 - min_max_safe(material["co2_emission_per_kg_estimated_"])
bio_norm = min_max_safe(material["biodegradation_time_days_"])
recycle_norm = min_max_safe(material["recyclability_"])

material["co2_impact_index"] = (
    0.4 * co2_norm +
    0.3 * bio_norm +
    0.3 * recycle_norm
) * 100

material["co2_impact_index"] = material["co2_impact_index"].clip(0, 100)




In [7]:
# cell 6 Cost Efficiency Index
cost_norm = 1 - min_max_safe(material["cost_per_unit_usd_"])
weight_norm = min_max_safe(material["total_material_weight_tons_"])
strength_norm = min_max_safe(material["strength"])

material["cost_efficiency_index"] = (
    0.4 * cost_norm +
    0.3 * weight_norm +
    0.3 * strength_norm
) * 100

material["cost_efficiency_index"] = material["cost_efficiency_index"].clip(0, 100)




In [9]:
#cell 7 Material Suitability Score
strength_norm_mss = min_max_safe(material["strength"])
weight_norm_mss = min_max_safe(material["total_material_weight_tons_"])

# Base numeric suitability score
base_mss = (
    0.6 * strength_norm_mss +
    0.4 * weight_norm_mss
)

# Industry boost (normalized influence)
industry_boost = {
    "Pharmacy": 1.0,
    "Electronics": 0.8,
    "Household": 0.5,
    "Apparel": 0.4
}

industry_factor = material["recommended_packaging_use_cases"].map(industry_boost).fillna(0.3)

# Final MSS
material["material_suitability_score"] = (
    0.85 * base_mss +
    0.15 * industry_factor
) * 100

material["material_suitability_score"] = material["material_suitability_score"].clip(0, 100)



In [10]:
#cell 8
material[
    ["co2_impact_index",
     "cost_efficiency_index",
     "material_suitability_score"]
].describe()



,co2_impact_index,cost_efficiency_index,material_suitability_score
count,403.000000,403.000000,403.000000
mean,57.930707,50.887692,42.594054
std,8.646935,5.629781,16.140727
min,34.203303,38.619549,7.056701
25%,51.423441,46.421150,31.544674
50%,57.512921,51.272128,44.225086
75%,66.195777,54.362275,51.180412
max,70.023144,66.190213,88.441581


In [12]:
#cell 9 Save engineered features
material.to_csv("../data/final/processed/material_engineered.csv", index=False)
